In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import produce_na
from imputation import imputation, get_rmse

np.random.seed(42)

In [2]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

experiment = Experiment(
    api_key=os.environ.get("COMET_API_KEY"),
    project_name="missing-data-imputation",
    workspace=None
)
experiment.set_name("Comparison_Run_v1")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/4b2d839e13b44dea999c9aa0fa4030d5



In [6]:
results_buffer = [] # список для хранения результатов, чтобы потом сохранить в эксель

# datasets = ["housing", "adult", "AirQuality"]
datasets = ["housing"]
# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": "population"
}
#  глобальный таргет
global_target = {
    "housing": "median_house_value"
}
ratios = [0.05, 0.20, 0.50]
# mechanisms = ["MCAR", "MAR", "MNAR"]
mechanisms = ["MCAR"]
# methods = ["Mean", "KNN", "MICE", "MissForest"]
methods = ["Simple", "KNN"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"strategy": "mean"},
        {"strategy": "median"},
    ],
    "KNN": [
        {"n_neighbors": 3},
        {"n_neighbors": 5},
        {"n_neighbors": 7},
    ],
}

# ЦИКЛ ПО ДАТАСЕТАМ
for dataset_name in datasets:
    file_path = Path("..") / f"data/processed/{dataset_name}/ground_truth.csv"
    df = pd.read_csv(file_path)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    # ЦИКЛ ПО МЕТОДАМ ВОЗНИКНОВЕНИЯ ПРОПУСКОВ
    for mechanism in mechanisms:
        # ЦИКЛ ПО КОЛИЧЕСТВУ ПРОПУСКОВ
        for ratio in ratios:
            # генерация пропусков
            df_incomplete, mask = produce_na(
                data=df, 
                target_col=[columns_target[dataset_name]], 
                mechanism=mechanism, 
                ratio=ratio)
            df_incomplete_numeric = df_incomplete.select_dtypes(include='number')
            # ЦИКЛ ПО МЕТОДАМ ВОССТАНОВЛЕНИЯ
            for method, param_list in EXPERIMENT_CONFIG.items():
                # ЦИКЛ ПО ГИПЕРПАРАМЕТРАМ
                for params in param_list:
                    start = time.perf_counter()
                    imputed_data = imputation(df_incomplete=df_incomplete_numeric,
                                            method=method,
                                            **params)
                    end = time.perf_counter()
                    rmse_score = get_rmse(df.loc[mask, columns_target[dataset_name]], imputed_data.loc[mask, columns_target[dataset_name]])
                    # model_accuracy = ... (метрика модели, если есть)
                    time_taken = end - start

                    # ЛОГИРОВАНИЕ
                    current_result = {
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "missing_ratio": ratio,
                        "imputation_method": method,
                        "params": params,
                        "rmse": rmse_score,
                        "time_sec": time_taken,
                    }

                    results_buffer.append(current_result)

                    # Логируем параметры эксперимента
                    experiment.log_parameters({
                        "dataset": dataset_name,
                        "mechanism": mechanism,
                        "missing_ratio": ratio,
                        "imputation_method": method,
                        **params,
                    })

                    # Логируем метрики
                    experiment.log_metrics(
                        {
                            "RMSE": rmse_score,
                            "Time_Sec": time_taken,
                        },
                        step=int(ratio * 100),
                    )

                    print(f"Done: {dataset_name} | {method} | {ratio}")

    df_results = pd.DataFrame(results_buffer)
    df_results.to_excel("experiment_results_backup.xlsx", index=False)


Done: housing | Simple | 0.05
Done: housing | Simple | 0.05
Done: housing | KNN | 0.05
Done: housing | KNN | 0.05
Done: housing | KNN | 0.05
Done: housing | Simple | 0.2
Done: housing | Simple | 0.2
Done: housing | KNN | 0.2
Done: housing | KNN | 0.2
Done: housing | KNN | 0.2
Done: housing | Simple | 0.5
Done: housing | Simple | 0.5
Done: housing | KNN | 0.5
Done: housing | KNN | 0.5
Done: housing | KNN | 0.5


In [7]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : Comparison_Run_v1
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/4b2d839e13b44dea999c9aa0fa4030d5
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     RMSE [15]     : (404.7479800840498, 1201.2007804601083)
COMET INFO:     Time_Sec [15] : (0.00294116600707639, 3.2927512919995934)
COMET INFO:   Others:
COMET INFO:     Name : Comparison_Run_v1
COMET INFO:   Parameters:
COMET INFO:     add_indicator       : False
COMET INFO:     copy                : True
COMET INFO:     dataset             : housing
COMET INFO:     fill_value          : None
COMET INFO:     imputation_method   : KNN
COMET INFO:    